<a href="https://colab.research.google.com/github/lipchenko/machine_learning_lipchenko/blob/main/cnn_Lipchenko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Домашнее задание: CNN для классификации описаний книг
Задача

Многоклассовая классификация писаний книг на сайте books.toscrape.  Вид сверточной нейросети - Text Cnn.
Источник данных

Сайт: https://books.toscrape.com/

Разметка - определение тональности при помощи sentiment blob.
Сбор датасета - библиотеки requests и BeautifulSoup

Модель для классификации реализована при помощи Pytorch. Модель - TextCNN


#Парсинг текста

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time

#добавляем ссылку на сайт books.torscape
BASE_URL = "https://books.toscrape.com/catalogue/"
OUTPUT_FILE = "books_700_with_description.csv"
LIMIT = 700
DELAY_PAGES = 0.5
DELAY_BOOKS = 0.3

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
#преобразуем звезды на сайте в рейтинг
def parse_rating(class_name):
    mapping = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    return mapping.get(class_name, 0)
#функция для извлечения описания книги
def get_book_description(book_url):

    try:
        resp = requests.get(book_url, headers=headers, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        # Вариант 1: div#product_description + следующий p
        desc_div = soup.find("div", id="product_description")
        if desc_div:
            desc_p = desc_div.find_next("p")
            if desc_p:
                return desc_p.text.strip()
        # Вариант 2: meta-тег description
        meta_desc = soup.find("meta", attrs={"name": "description"})
        if meta_desc and meta_desc.get("content"):
            return meta_desc["content"].strip()
        return "Описание не найдено"
    except Exception:
        return "Ошибка загрузки"
#добавила два способа поиска описания, чтобы сработал хотя бы один из них
#парсинг страницы с книгами для всех атрибутов: название, цена, рейтинг, наличие, ссылка, описание
def get_books_from_page(url):

    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.find_all("article", class_="product_pod")
    books = []
    for article in articles:
        title_tag = article.h3.a
        title = title_tag.get("title") if title_tag else "Без названия"

        price_tag = article.find("p", class_="price_color")
        price = price_tag.text.strip() if price_tag else "N/A"

        rating_tag = article.find("p", class_="star-rating")
        rating_class = rating_tag.get("class")[1] if rating_tag and len(rating_tag.get("class")) > 1 else "Zero"
        rating = parse_rating(rating_class)

        availability_tag = article.find("p", class_="instock availability")
        availability = availability_tag.text.strip() if availability_tag else "Неизвестно"

        relative_link = title_tag.get("href") if title_tag else ""
        book_url = url.rsplit("/", 1)[0] + "/" + relative_link if relative_link else ""

        description = get_book_description(book_url) if book_url else "Нет URL"
        time.sleep(DELAY_BOOKS)

        books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "book_url": book_url,
            "description": description
        })
    return books

# ------ Основной сбор с ограничением ------
all_books = []
page = 1

while len(all_books) < LIMIT:
    url = f"{BASE_URL}page-{page}.html"
    print(f"Страница {page} (уже собрано: {len(all_books)})")

    books_on_page = get_books_from_page(url)
    if not books_on_page:
        print("Книги не найдены")
        break

    #добавила лимит 700 книг
    for book in books_on_page:
        if len(all_books) >= LIMIT:
            break
        all_books.append(book)

    #проверка следующих страниц
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        if not soup.find("li", class_="next"):
            print("🔚 Достигнут конец каталога.")
            break
    except Exception as e:
        print(f"Ошибка при проверке следующей страницы: {e}")
        break

    page += 1
    time.sleep(DELAY_PAGES)

print(f"\nСобрано {len(all_books)} книг (цель: {LIMIT}).")

if all_books:
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["title", "price", "rating", "availability", "book_url", "description"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_books)
    print(f"Данные сохранены в {OUTPUT_FILE}")
else:
    print("Не удалось собрать ни одной книги.")

Страница 1 (уже собрано: 0)
Страница 2 (уже собрано: 20)
Страница 3 (уже собрано: 40)
Страница 4 (уже собрано: 60)
Страница 5 (уже собрано: 80)
Страница 6 (уже собрано: 100)
Страница 7 (уже собрано: 120)
Страница 8 (уже собрано: 140)
Страница 9 (уже собрано: 160)
Страница 10 (уже собрано: 180)
Страница 11 (уже собрано: 200)
Страница 12 (уже собрано: 220)
Страница 13 (уже собрано: 240)
Страница 14 (уже собрано: 260)
Страница 15 (уже собрано: 280)
Страница 16 (уже собрано: 300)
Страница 17 (уже собрано: 320)
Страница 18 (уже собрано: 340)
Страница 19 (уже собрано: 360)
Страница 20 (уже собрано: 380)
Страница 21 (уже собрано: 400)
Страница 22 (уже собрано: 420)
Страница 23 (уже собрано: 440)
Страница 24 (уже собрано: 460)
Страница 25 (уже собрано: 480)
Страница 26 (уже собрано: 500)
Страница 27 (уже собрано: 520)
Страница 28 (уже собрано: 540)
Страница 29 (уже собрано: 560)
Страница 30 (уже собрано: 580)
Страница 31 (уже собрано: 600)
Страница 32 (уже собрано: 620)
Страница 33 (уже собран

#Автоматическая разметка датасета

In [ ]:
import pandas as pd
from textblob import TextBlob

# Загрузка датасета
df = pd.read_csv("books_700_with_description.csv")

# размечаем тональность описания при помощи TextBlob
def get_sentiment(text):
    blob = TextBlob(str(text))
    return blob.sentiment.polarity, blob.sentiment.subjectivity

df['polarity'], df['subjectivity'] = zip(*df['description'].apply(get_sentiment))

# Добавила категорию тональности positive, negative, neutral
def sentiment_label(polarity):
    if polarity > 0.1:
        return 'positive'
    elif polarity < -0.1:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_label'] = df['polarity'].apply(sentiment_label)

# сохраняем все в файл csv
df.to_csv("books_annotated.csv", index=False)
print("Разметка тональности завершена!")

Разметка тональности завершена!


#Загрузка датасета

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from collections import Counter
import re
import os

CSV_PATH = "/content/books_annotated.csv"

df = pd.read_csv(CSV_PATH)

# Проверка на наличие колонок "description" и "sentiment_label"
if 'description' not in df.columns or 'sentiment_label' not in df.columns:
    raise ValueError("Нет колонок 'description' и 'sentiment_label'")

# Берём только нужные колонки и удаляем строки с пустыми текстами
df = df[['description', 'sentiment_label']].dropna()
df = df[df['description'].str.strip() != '']

# Преобразуем метки: positive - 1, negative - 0
df['label'] = df['sentiment_label'].map({'positive': 2, 'neutral': 1, 'negative': 0})
# Удаляем строки, где метка не распозналась
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(f"Загружено {len(df)} записей. Распределение классов:")
print(df['label'].value_counts())


Загружено 699 записей. Распределение классов:
label
2    433
1    235
0     31
Name: count, dtype: int64


In [ ]:
#предобработка текста
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Оставляем только буквы и пробелы
    text = re.sub(r'[^a-zа-яё\s]', '', text)
    return text

df['clean_text'] = df['description'].apply(clean_text)
df = df[df['clean_text'].str.len() > 0]

X = df['clean_text'].values
y = df['label'].values

In [ ]:
#разделение валидационные, тестовые и тренировочные данные
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


Train: 489, Val: 105, Test: 105


In [ ]:
#токенизируем и создаем мсловарь
word_counts = Counter()
for text in X_train:
    words = text.split()
    word_counts.update(words)

vocab_size = 10000
most_common = word_counts.most_common(vocab_size - 2)
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, _ in most_common:
    word2idx[word] = len(word2idx)

max_len = 100

def text_to_seq(text):
    words = text.split()
    seq = [word2idx.get(word, 1) for word in words]
    if len(seq) > max_len:
        seq = seq[:max_len]
    else:
        seq = seq + [0] * (max_len - len(seq))
    return seq

X_train_seq = np.array([text_to_seq(t) for t in X_train])
X_val_seq   = np.array([text_to_seq(t) for t in X_val])
X_test_seq  = np.array([text_to_seq(t) for t in X_test])


In [ ]:
#подготовка загрузчиков данных
batch_size = 64
train_dataset = TensorDataset(torch.LongTensor(X_train_seq), torch.LongTensor(y_train))
val_dataset   = TensorDataset(torch.LongTensor(X_val_seq),   torch.LongTensor(y_val))
test_dataset  = TensorDataset(torch.LongTensor(X_test_seq),  torch.LongTensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size)

In [ ]:
#определение модели TextCNN
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes, num_classes, dropout):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        conv_outs = []
        for conv in self.convs:
            conv_out = torch.relu(conv(x))
            pooled = torch.max_pool1d(conv_out, conv_out.size(2)).squeeze(2)
            conv_outs.append(pooled)
        x = torch.cat(conv_outs, dim=1)
        x = self.dropout(x)
        return self.fc(x)

# Параметры модели
embed_dim = 100
num_filters = 128
filter_sizes = [3, 4, 5]
dropout = 0.5
num_classes = 3
learning_rate = 0.001
num_epochs = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

model = TextCNN(vocab_size, embed_dim, num_filters, filter_sizes, num_classes, dropout).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


Используется устройство: cpu


Слои модели: Эмбеддинговый слой (embed_dim), параллельные сверточные слои, операция глобального максипулинга, конкатенация признаков, полносвязный слой, выходной слой.
В эмбединоговом слое каждый токен входной поседовательности в плотный вектор размерности embed_dim = 100. В параллельном сверточном слое применяется три слоя с ядрами [3, 4, 5]. Каждая свертка использует 128 фильтров. В слое максипулинга выбирается максимальное значение фильтра по всей длине, чтобы выявить наиболее значимый признак для каждого фильтра и привести все выходы к фиксированной длине. Выходы всех светрочных слов объединяются в в один вектор размерности. После объединения применяетс яслой с отсечением (Dropout = 0.5) для предотвращения переобучения, а затем линейный слой, который преобразует вектор размерности в num_classes выходных нейронов. В выходном слое CrossEntropyLoss преобразует выходы полносвязного слоя в вероятности принадлежности текста к каждому из классов.

#Почему выбрана такая модель?

TextCNN хорошо улавливает локальные закономерности в тексте (например, сочетания слов, характерные для позитивных или негативных отзывов) и не требует сложной предобработки. Модель хорошо работает на небольших размерах данных (700 записей в датасете), а регуляризация (Dropout) помогает избежать переобучения.Размеры ядер позволяет учитывать словосочетания различной длины, это повыщает качество классификации.

In [ ]:
#Обучение модели
best_val_acc = 0.0
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Валидация
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    val_acc = correct / total

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print("Модель сохранена")

Epoch 1/20, Loss: 1.0284, Val Acc: 0.6190
Модель сохранена
Epoch 2/20, Loss: 0.8221, Val Acc: 0.6190
Epoch 3/20, Loss: 0.6219, Val Acc: 0.6190
Epoch 4/20, Loss: 0.5850, Val Acc: 0.6190
Epoch 5/20, Loss: 0.5089, Val Acc: 0.6190
Epoch 6/20, Loss: 0.4130, Val Acc: 0.6190
Epoch 7/20, Loss: 0.3455, Val Acc: 0.6190
Epoch 8/20, Loss: 0.2789, Val Acc: 0.6190
Epoch 9/20, Loss: 0.2561, Val Acc: 0.6190
Epoch 10/20, Loss: 0.2163, Val Acc: 0.6190
Epoch 11/20, Loss: 0.1887, Val Acc: 0.6190
Epoch 12/20, Loss: 0.1734, Val Acc: 0.6190
Epoch 13/20, Loss: 0.1503, Val Acc: 0.6190
Epoch 14/20, Loss: 0.1369, Val Acc: 0.6190
Epoch 15/20, Loss: 0.1060, Val Acc: 0.6190
Epoch 16/20, Loss: 0.1045, Val Acc: 0.6190
Epoch 17/20, Loss: 0.1066, Val Acc: 0.6190
Epoch 18/20, Loss: 0.0938, Val Acc: 0.6190
Epoch 19/20, Loss: 0.0916, Val Acc: 0.6190
Epoch 20/20, Loss: 0.0800, Val Acc: 0.6190


In [ ]:
# Оценка тестовой выборки
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Метрики для многоклассовой классификации
acc = accuracy_score(all_labels, all_preds)

prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

print(f"Accuracy:  {acc:.4f}")
print(f"Weighted Precision: {prec:.4f}")
print(f"Weighted Recall:    {rec:.4f}")
print(f"Weighted F1-score:  {f1:.4f}")

# Детальный отчёт по каждому классу
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['negative', 'neutral', 'positive']))

Accuracy:  0.6190
Weighted Precision: 0.3832
Weighted Recall:    0.6190
Weighted F1-score:  0.4734

Classification Report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         5
     neutral       0.00      0.00      0.00        35
    positive       0.62      1.00      0.76        65

    accuracy                           0.62       105
   macro avg       0.21      0.33      0.25       105
weighted avg       0.38      0.62      0.47       105



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

#Анализ
Модель достигла максимальной точности для класса positive, но игнорирует остальные классы. Это могло произойти из-за несбалансированности датасета. Возможно стоит пересмотреть способ разметки данных более удачного результата. Также можно пересмотреть архитектуру: использовать предобученные эмбединги как Word2Vec и FastText, увеличить количество фильтров или изменить размер ядер.